# Auditoria de negativos difíceis

Este notebook examina contextos internos construídos após remover o documento gold. A prevalência é artificialmente zero; os resultados descrevem estresse, não calibração operacional.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'results').is_dir() else Path.cwd().parent
SCENARIOS = ROOT / 'data/interim/scenarios_hard_v1'
RUN = ROOT / 'results/runs/20260721-023427_hard-stress-k5_594f6518'


In [ ]:
scenario_manifest = json.loads((SCENARIOS / 'manifest.json').read_text())
run_manifest = json.loads((RUN / 'manifest.json').read_text())
assert scenario_manifest['final_test_used'] is False
assert run_manifest['final_test_used'] is False
print('Escopo confirmado: somente splits internos.')


In [ ]:
scenarios = pd.read_parquet(SCENARIOS / 'scenarios.parquet')
assert scenarios['included_chunk_ids'].map(len).eq(5).all()
assert scenarios.apply(lambda r: r['relevant_document_id'] not in r['included_document_ids'], axis=1).all()
assert not scenarios['evidence_sufficient'].any()
scenarios.groupby(['research_split', 'scenario_type']).size().unstack(fill_value=0)


In [ ]:
metrics = json.loads((RUN / 'metrics.json').read_text())
pd.DataFrame({
    'falsa_autorização': metrics['overall']['false_acceptance_rate'],
    'IC95% inferior': {k: v['ci_lower'] for k, v in metrics['bootstrap']['false_acceptance_rate'].items()},
    'IC95% superior': {k: v['ci_upper'] for k, v in metrics['bootstrap']['false_acceptance_rate'].items()},
}).round(3)


In [ ]:
paired = pd.read_parquet(RUN / 'paired_natural_comparison.parquet')
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(paired['natural_probability'], paired['probability'], alpha=0.7)
ax.plot([0, 1], [0, 1], '--', color='0.5')
ax.set(xlabel='Probabilidade no contexto natural', ylabel='Após remoção do gold', xlim=(0, 1), ylim=(0, 1))
ax.set_title('Mudança pareada de confiança — seleção interna, k=5')
ax.grid(alpha=0.2);


## Limite de interpretação

A ausência do documento anotado confirma a ausência da evidência gold, não a inexistência de qualquer evidência equivalente em documentos não anotados. Candidatos genéricos de conflito técnico permanecem sujeitos a revisão manual.